In [1]:
basein = "bioemu_test_150_batch75"

In [2]:
import os, sys, subprocess
import pandas as pd
from tqdm.notebook import tqdm

In [3]:
sys.path.append("..")

In [4]:
from predict import \
get_cif, Cif, Site, get_pockets, Pocket, \
get_pdb_features, get_pockets_features, \
prepare_data, model, view_pockets
# get_features

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/autogluon/common/utils/utils.py:78: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [5]:
site_residues = [{"label_asym_id": "A", "label_seq_id": seqnum} for seqnum in ('73', '77', '78', '80', '81', '84', '85', '88', '109', '110', '111', '112', '113', '114', '232', '235', '236', '238', '239', '240', '241', '243', '248', '250', '251', '252', '253', '254', '255', '257', '258', '261', '262')]

## sc

In [13]:
indir = f"{basein}/ttclust_relax_nomd"

In [17]:
overlaps = {}

for i in range(4):
    pdbf = f"{indir}/frame_{i}.pdb"
    path = f"{indir}/predict_{i}"
    predsf = f"{path}/preds.pkl"
    
    clean_pdb = get_cif(file=pdbf, path=path)
    os.system(f"ln -s {clean_pdb.entry_id}_updated.cif {path}/{clean_pdb.entry_id}.cif")
    clean_pdb.filename = f"{path}/{clean_pdb.entry_id}.cif"

    if not os.path.isfile(predsf):
        pockets = get_pockets(
            clean_pdb,
            out=sys.stdout,
            path=path
        )
        pockets["pdb"] = clean_pdb.entry_id
    
        os.makedirs(f"{path}/features/{clean_pdb.entry_id}", exist_ok=True)
        os.system(f"ln -s ../../../../relax_nomd/predict_0/features/frame_0/HHBlitsF.pkl {path}/features/{clean_pdb.entry_id}/HHBlitsF.pkl")
    
        for feat in ['features', 'transferentropy', 'hhblits']:
            subprocess.run(f"python ../gradio/{feat}.py {clean_pdb.entry_id} {path}", shell=True, check=True)
    
        features = get_pdb_features(
            clean_pdb,
            sites = [pd.DataFrame(columns=clean_pdb.residues.columns),],
            features_path = path
        )
    
        pockets_features = get_pockets_features(
            clean_pdb,
            pockets,
            features,
            path=path
        )
    
        data = prepare_data(pockets_features)
    
        preds = model.predict_proba(data)[[1]].sort_values(1, ascending=False).rename(columns={1: "Allosteric score"})
        preds.index = preds.index.map(lambda x: x.split("_")[-1])
        
        preds.to_pickle(predsf)
    else:
        preds = pd.read_pickle(predsf)

    res_site = Site(
        pdb=clean_pdb,
        residues=site_residues,
        only_protein=True
    )

    topp = preds.iloc[0].name
    merge = (
        res_site.residues
        .merge(
            Pocket(f"{path}/{clean_pdb.entry_id}/{clean_pdb.entry_id}_out/pockets/{topp}_atm.cif")
            .residues
        )
    )
    overlaps[f"frame_{i}"] = {
        "pocket": topp,
        "overlap": len(merge) / len(res_site.residues),
        "merge": merge
    }

ln: failed to create symbolic link 'bioemu_test_150_batch75/ttclust_relax_nomd/predict_0/frame_0.cif': File exists
/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags 
core.init: Rosetta version: PyRosetta4.conda.ubuntu.cxx11thread.se

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

In [18]:
overlaps

{'frame_0': {'pocket': 'pocket1',
  'overlap': 0.30303030303030304,
  'merge':   label_comp_id label_asym_id pdbx_PDB_ins_code auth_asym_id  \
  0           TRP             A                 ?            A   
  1           ALA             A                 ?            A   
  2           LEU             A                 ?            A   
  3           MET             A                 ?            A   
  4           CYS             A                 ?            A   
  5           HIS             A                 ?            A   
  6           ARG             A                 ?            A   
  7           LEU             A                 ?            A   
  8           PHE             A                 ?            A   
  9           TYR             A                 ?            A   
  
    pdbx_PDB_model_num auth_comp_id auth_seq_id label_entity_id label_seq_id  
  0                  1          TRP          72               1           73  
  1                  1          ALA 

In [21]:
i = 2
pocket = 1
v = view_pockets(
    Cif(f"frame_{i}", f"{indir}/predict_{i}/frame_{i}_updated.cif"),
    pockets={f"pocket{pocket}": {"color": "green"}},
    site_residues=pd.DataFrame(site_residues),#res_site.residues,
    path=f"{indir}/predict_{i}"
)
v

PDBeMolstar(bg_color='#F7F7F7', color_data={'data': [{'struct_asym_id': 'A', 'representation': 'cartoon', 'rep…

### Results

- Worse than foldseek.

## sc + minim

### Loop

In [22]:
indir = f"{basein}/ttclust_relax_minim_parallel"

In [23]:
overlaps = {}

for i in range(4):
    pdbf = f"{indir}/frame_{i}.pdb"
    path = f"{indir}/predict_{i}"
    predsf = f"{path}/preds.pkl"
    
    clean_pdb = get_cif(file=pdbf, path=path)
    os.system(f"ln -s {clean_pdb.entry_id}_updated.cif {path}/{clean_pdb.entry_id}.cif")
    clean_pdb.filename = f"{path}/{clean_pdb.entry_id}.cif"

    if not os.path.isfile(predsf):
        pockets = get_pockets(
            clean_pdb,
            out=sys.stdout,
            path=path
        )
        pockets["pdb"] = clean_pdb.entry_id
    
        os.makedirs(f"{path}/features/{clean_pdb.entry_id}", exist_ok=True)
        os.system(f"ln -s ../../../../relax_nomd/predict_0/features/frame_0/HHBlitsF.pkl {path}/features/{clean_pdb.entry_id}/HHBlitsF.pkl")
    
        for feat in ['features', 'transferentropy', 'hhblits']:
            subprocess.run(f"python ../gradio/{feat}.py {clean_pdb.entry_id} {path}", shell=True, check=True)
    
        features = get_pdb_features(
            clean_pdb,
            sites = [pd.DataFrame(columns=clean_pdb.residues.columns),],
            features_path = path
        )
    
        pockets_features = get_pockets_features(
            clean_pdb,
            pockets,
            features,
            path=path
        )
    
        data = prepare_data(pockets_features)
    
        preds = model.predict_proba(data)[[1]].sort_values(1, ascending=False).rename(columns={1: "Allosteric score"})
        preds.index = preds.index.map(lambda x: x.split("_")[-1])
        
        preds.to_pickle(predsf)
    else:
        preds = pd.read_pickle(predsf)

    res_site = Site(
        pdb=clean_pdb,
        residues=site_residues,
        only_protein=True
    )

    topp = preds.iloc[0].name
    merge = (
        res_site.residues
        .merge(
            Pocket(f"{path}/{clean_pdb.entry_id}/{clean_pdb.entry_id}_out/pockets/{topp}_atm.cif")
            .residues
        )
    )
    overlaps[f"frame_{i}"] = {
        "pocket": topp,
        "overlap": len(merge) / len(res_site.residues),
        "merge": merge
    }

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

In [24]:
overlaps

{'frame_0': {'pocket': 'pocket1',
  'overlap': 0.36363636363636365,
  'merge':    label_comp_id label_asym_id pdbx_PDB_ins_code auth_asym_id  \
  0            LYS             A                 ?            A   
  1            ALA             A                 ?            A   
  2            GLY             A                 ?            A   
  3            MET             A                 ?            A   
  4            CYS             A                 ?            A   
  5            HIS             A                 ?            A   
  6            VAL             A                 ?            A   
  7            LEU             A                 ?            A   
  8            GLN             A                 ?            A   
  9            GLN             A                 ?            A   
  10           LEU             A                 ?            A   
  11           PHE             A                 ?            A   
  
     pdbx_PDB_model_num auth_comp_id auth_seq_id 

In [27]:
i = 2
pocket = 2
v = view_pockets(
    Cif(f"frame_{i}", f"{indir}/predict_{i}/frame_{i}_updated.cif"),
    pockets={f"pocket{pocket}": {"color": "green"}},
    site_residues=pd.DataFrame(site_residues),#res_site.residues,
    path=f"{indir}/predict_{i}"
)
v

PDBeMolstar(bg_color='#F7F7F7', color_data={'data': [{'struct_asym_id': 'A', 'representation': 'cartoon', 'rep…

### Results

- Worse

## sc + minim + equil

### Loop

In [28]:
indir = f"{basein}/ttclust_relax_equil_parallel"

In [30]:
overlaps = {}

for i in tqdm(range(4)):
    pdbf = f"{indir}/frame_{i}.pdb"
    path = f"{indir}/predict_{i}"
    predsf = f"{path}/preds.pkl"
    
    clean_pdb = get_cif(file=pdbf, path=path)
    os.system(f"ln -s {clean_pdb.entry_id}_updated.cif {path}/{clean_pdb.entry_id}.cif")
    clean_pdb.filename = f"{path}/{clean_pdb.entry_id}.cif"

    if not os.path.isfile(predsf):
        pockets = get_pockets(
            clean_pdb,
            out=sys.stdout,
            path=path
        )
        pockets["pdb"] = clean_pdb.entry_id
    
        os.makedirs(f"{path}/features/{clean_pdb.entry_id}", exist_ok=True)
        os.system(f"ln -s ../../../../relax_nomd/predict_0/features/frame_0/HHBlitsF.pkl {path}/features/{clean_pdb.entry_id}/HHBlitsF.pkl")
    
        for feat in ['features', 'transferentropy', 'hhblits']:
            subprocess.run(f"python ../gradio/{feat}.py {clean_pdb.entry_id} {path}", shell=True, check=True)
    
        features = get_pdb_features(
            clean_pdb,
            sites = [pd.DataFrame(columns=clean_pdb.residues.columns),],
            features_path = path
        )
    
        pockets_features = get_pockets_features(
            clean_pdb,
            pockets,
            features,
            path=path
        )
    
        data = prepare_data(pockets_features)
    
        preds = model.predict_proba(data)[[1]].sort_values(1, ascending=False).rename(columns={1: "Allosteric score"})
        preds.index = preds.index.map(lambda x: x.split("_")[-1])
        
        preds.to_pickle(predsf)
    else:
        preds = pd.read_pickle(predsf)

    res_site = Site(
        pdb=clean_pdb,
        residues=site_residues,
        only_protein=True
    )

    topp = preds.iloc[0].name
    merge = (
        res_site.residues
        .merge(
            Pocket(f"{path}/{clean_pdb.entry_id}/{clean_pdb.entry_id}_out/pockets/{topp}_atm.cif")
            .residues
        )
    )
    overlaps[f"frame_{i}"] = {
        "pocket": topp,
        "overlap": len(merge) / len(res_site.residues),
        "merge": merge
    }

  0%|          | 0/4 [00:00<?, ?it/s]

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

/home/fnerin/miniconda3/envs/allopockets/lib/python3.11/site-packages/prody/utilities/misctools.py:424: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python311.Release 2025.24+release.8e1e5e54f047b0833dcf760a5cd5d3ce94d63938 2025-06-06T09:20:57] retrieved from: http://www.pyrosetta.org
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00
Reading PDB file...         ━━━━━━━━━━━━━━━━━━━━━━━

In [31]:
overlaps

{'frame_0': {'pocket': 'pocket1',
  'overlap': 0.3333333333333333,
  'merge':    label_comp_id label_asym_id pdbx_PDB_ins_code auth_asym_id  \
  0            TRP             A                 ?            A   
  1            HIS             A                 ?            A   
  2            LEU             A                 ?            A   
  3            THR             A                 ?            A   
  4            GLN             A                 ?            A   
  5            VAL             A                 ?            A   
  6            LEU             A                 ?            A   
  7            MET             A                 ?            A   
  8            HIS             A                 ?            A   
  9            PRO             A                 ?            A   
  10           LEU             A                 ?            A   
  
     pdbx_PDB_model_num auth_comp_id auth_seq_id label_entity_id label_seq_id  
  0                   1          TRP 

In [33]:
i = 3
pocket = 1
v = view_pockets(
    Cif(f"frame_{i}", f"{indir}/predict_{i}/frame_{i}_updated.cif"),
    pockets={f"pocket{pocket}": {"color": "green"}},
    site_residues=pd.DataFrame(site_residues),#res_site.residues,
    path=f"{indir}/predict_{i}"
)
v

PDBeMolstar(bg_color='#F7F7F7', color_data={'data': [{'struct_asym_id': 'A', 'representation': 'cartoon', 'rep…

### Results

- Now there's one pocket that crosses the orthosteric and allosteric :-)